# omnidroid, headless, in a notebook

This notebook builds [omnidroid](https://github.com/devflaryn/omnidroid) from source, starts
Roblox (the real arm64 Android APK, run by omnidroid without a VM) **with no screen**, joins a
place, waits until the world has loaded, takes a **screenshot as proof**, shows it here and
prints it as a `data:` URL you can paste into a browser's address bar.

It runs on **Google Colab**, **Kaggle**, **Modal notebooks** and any **Ubuntu/Debian Linux**
Jupyter. It finds out where it is running; you only fill in the parameters cell.

| | Colab | Kaggle | Modal notebook | your Linux |
|---|---|---|---|---|
| APK | Google Drive `MyDrive/omnidroid/*.apk`, or uploaded to `/content` | a private dataset (`/kaggle/input/<name>/*.apk`) | a Volume attached at `/mnt/<volume>` | `APK_PATH` |
| cookie | Secrets panel (key icon) `ROBLOSECURITY` | Add-ons > Secrets `ROBLOSECURITY` | a Modal Secret with `ROBLOSECURITY` | env `ROBLOSECURITY` or `COOKIE_FILE` |
| build cache | Drive `MyDrive/omnidroid/cache` | output -> dataset | the Volume | `~/omnidroid-nb/cache` |
| needs | nothing else | **Settings > Internet: on** | an image with apt (the default works) | apt (root or `sudo -n`), or the packages already present |

**The APK must be the stock Roblox build** (2.738.1397 from the Play Store, sha256 `bbe00ae3...`,
set in `APK_SHA256`): the notebook checks the hash and refuses any other file, because a modified
APK given a cookie can take the account.

**No display at all:** `omnidroid --no-window` draws into an off-screen EGL surface, headless
(the game runs and presents frames while nothing is drawn); a `screenshot` command renders that
one frame for real. **GPU or CPU:** with a GPU (T4, L4, A100, H100...) whose EGL library the
platform exposes, it renders through NVIDIA's EGL; with no GPU -- or one whose graphics
libraries are not exposed -- through Mesa **llvmpipe** on the CPU, which is enough for the game
and a screenshot (measured: in the world, ~5 frames/s drawn on 4 old cores).

**Time** (measured on a 4-core Ubuntu 22.04 and 26.04, CPU only): packages ~2 min; Rust ~0.5 min;
the first build **~5-6 min** on 4 cores (expect ~10-15 on Colab's 2 vCPUs) -- cached afterwards,
~0.1 GiB; start to the world on screen **~3-4 min** on llvmpipe (sign-in ~20 s, the place loads
~1.5 min in, then its loading screen for another ~1.5 min), longer on 2 slow CPUs.
Memory: one instance holds ~2.5-4 GiB; 12 GiB machines are fine.

> ## ⚠️ The cookie: read this before running
> * The cookie is the account's **`.ROBLOSECURITY`** session. Anyone who has it *is* that account.
>   This notebook reads it from the platform's secret store, writes it to a file only this user
>   can read, and **never prints it**. Do not paste it into a cell.
> * **Roblox ends sessions whose cookie shows up from another country.** A cookie made in Turkey
>   and used from a US or European data centre (Colab, Kaggle and Modal all are) can be **killed
>   for good** -- you would have to sign in again to get a new one. Use an account that is made
>   or used from the region the notebook runs in, or a throwaway account. The notebook prints the
>   machine's exit country before it uses the cookie; set `ACCOUNT_COUNTRY` to make it **stop**
>   instead when they differ.
> * Joining a game needs outbound **UDP 49152-65535**. All three platforms allow it today.

**Run it:** fill in the parameters, then *Run all* (Colab: Runtime > Run all). The screenshot
is in the output of the **Screenshot** cell; the cells after it switch rendering off and on,
take more screenshots, and stop the game.

In [ ]:
# @title Parameters
# Source
REPO_URL = "https://github.com/devflaryn/omnidroid"  # @param {type:"string"}
BRANCH = "unified"  # @param {type:"string"}
SOURCE_TARBALL = ""  # @param {type:"string"}
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # @param {type:"string"}
RUST_TOOLCHAIN = "1.98.1"  # @param {type:"string"}
USE_BUILD_CACHE = True  # @param {type:"boolean"}
BUILD_JOBS = 0  # @param {type:"integer"}
# The app and the account
APK_PATH = ""  # @param {type:"string"}
# The stock Roblox 2.738.1397 APK. The run refuses any other file: a modified build with a cookie
# hands the account to whoever modified it. For another version, put its stock sha256 here.
APK_SHA256 = "bbe00ae306cc251c4ea55b7a932d9c524ecb0d6d9203c2a6161bcf0fae792742"  # @param {type:"string"}
COOKIE_SECRET = "ROBLOSECURITY"  # @param {type:"string"}
COOKIE_FILE = ""  # @param {type:"string"}
ACCOUNT_NAME = "notebook"  # @param {type:"string"}
ACCOUNT_COUNTRY = ""  # @param {type:"string"}
PLACE_ID = 8737899170  # @param {type:"integer"}
# How it runs
RENDERER = "auto"  # @param ["auto", "gpu", "cpu"]
START_HEADLESS = True  # @param {type:"boolean"}
WINDOW_SIZE = "1280x720"  # @param {type:"string"}
GRAPHICS_QUALITY = 1  # @param {type:"integer"}
SESSION_MINUTES = 120  # @param {type:"integer"}
JOIN_DELAY = 0  # @param {type:"integer"}
LOAD_TIMEOUT_MINUTES = 30  # @param {type:"integer"}
SETTLE_SECONDS = 60  # @param {type:"integer"}
MOUNT_DRIVE = True  # @param {type:"boolean"}
LAUNCH_PREFIX = ""  # @param {type:"string"}

# Every parameter can also be set from the environment as OMNI_NB_<NAME> (Modal notebooks,
# scripted runs): the environment wins.
import os as _os
for _name in [n for n in list(globals()) if n.isupper() and not n.startswith("_")]:
    _raw = _os.environ.get("OMNI_NB_" + _name)
    if _raw is None:
        continue
    _old = globals()[_name]
    if isinstance(_old, bool):
        globals()[_name] = _raw.strip().lower() in ("1", "true", "yes", "on")
    elif isinstance(_old, int):
        globals()[_name] = int(_raw)
    else:
        globals()[_name] = _raw
    print(f"{_name} = {globals()[_name]!r}  (from OMNI_NB_{_name})")

## 1. Where am I?
Detects the platform, the paths it uses, and the helpers the other cells share.

In [ ]:
import base64, collections, ctypes, glob, hashlib, io, json, os, platform, re, shlex, shutil
import signal, subprocess, sys, textwrap, time
from pathlib import Path


def detect_platform():
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.path.isdir("/kaggle/input"):
        return "kaggle"
    if os.environ.get("MODAL_IS_REMOTE") == "1" or os.environ.get("MODAL_TASK_ID"):
        return "modal"
    return "linux"


PLATFORM = os.environ.get("OMNI_NB_PLATFORM") or detect_platform()
IS_ROOT = os.geteuid() == 0
HOME = Path.home()


def _sudo_works():
    return shutil.which("sudo") is not None and subprocess.run(
        ["sudo", "-n", "true"], capture_output=True).returncode == 0


SUDO = [] if IS_ROOT else (["sudo", "-n"] if _sudo_works() else None)

# The source lives at one fixed absolute path per machine: the build records it (the gate reads
# files beside its own source), so a cached build is only valid at the same path.
SRC = Path(os.environ.get("OMNI_NB_SRC") or ("/opt/omnidroid" if IS_ROOT else HOME / "omnidroid-nb" / "src"))
WORK = Path(os.environ.get("OMNI_NB_WORK") or {
    "colab": "/content/omnidroid-run",
    "kaggle": "/kaggle/working/omnidroid-run",
    "modal": "/root/omnidroid-run",
}.get(PLATFORM, str(HOME / "omnidroid-nb" / "run")))
SHOTS = Path(os.environ.get("OMNI_NB_SHOTS") or {
    "colab": "/content", "kaggle": "/kaggle/working"}.get(PLATFORM, str(WORK / "shots")))
for d in (WORK, SHOTS):
    d.mkdir(parents=True, exist_ok=True)

CARGO_HOME = Path(os.environ.get("CARGO_HOME", HOME / ".cargo"))
os_release = dict(
    line.rstrip("\n").split("=", 1) for line in open("/etc/os-release") if "=" in line)
DISTRO = os_release.get("ID", "?").strip('"') + os_release.get("VERSION_ID", "").strip('"')
GLIBC = platform.libc_ver()[1]


def sh(cmd, env=None, cwd=None, check=True, show=None, quiet=False):
    """Run `cmd` (a list, or a string for bash), streaming its output. `show(line)` picks the lines
    printed; the last 80 lines are kept and printed if it fails."""
    shell = isinstance(cmd, str)
    proc = subprocess.Popen(cmd, shell=shell, executable="/bin/bash" if shell else None, env=env,
                            cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            errors="replace", bufsize=1)
    tail = collections.deque(maxlen=80)
    for line in proc.stdout:
        tail.append(line)
        if not quiet and (show is None or show(line)):
            print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        print("".join(tail))
        raise RuntimeError(f"command failed with exit code {code}: {cmd if shell else shlex.join(cmd)}")
    return code, "".join(tail)


def get_secret(name):
    """A secret by name: the environment first, then the platform's own secret store."""
    if not name:
        return None
    value = os.environ.get(name)
    if value:
        return value
    if PLATFORM == "colab":
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception as error:  # not defined, or this notebook was not given access
            print(f"(Colab secret {name!r}: {type(error).__name__})")
    if PLATFORM == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception as error:
            print(f"(Kaggle secret {name!r}: {type(error).__name__} -- attach it under Add-ons > Secrets)")
    return None


def elapsed(t0):
    s = int(time.time() - t0)
    return f"{s // 60}:{s % 60:02d}"


print(f"platform: {PLATFORM}   {os_release.get('PRETTY_NAME', '?')}   glibc {GLIBC}   python {platform.python_version()}")
print(f"user: {'root' if IS_ROOT else os.environ.get('USER', '?')}   package installs: "
      f"{'yes' if SUDO is not None else 'no (no root, no passwordless sudo)'}")
print(f"cpus: {os.cpu_count()}   memory: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30:.1f} GiB")
print(f"source: {SRC}\nwork:   {WORK}\nshots:  {SHOTS}")

## 2. System packages
Compilers, CMake, the X11/ALSA/Vulkan headers omnidroid builds against, Mesa (EGL, GLES,
llvmpipe, lavapipe). Where nothing can be installed (no root, no
passwordless sudo) it only checks what is already there.

In [ ]:
APT_PACKAGES = """
build-essential cmake ninja-build pkg-config git curl ca-certificates zstd xz-utils
libasound2-dev libx11-dev libxi-dev libxfixes-dev
libvulkan1 libvulkan-dev mesa-vulkan-drivers
libegl1 libgles2 libegl-mesa0 libgl1-mesa-dri libglvnd0
""".split()
if DISTRO == "ubuntu22.04":
    # Ubuntu 22.04's default g++ is 11; dynarmic is C++20, and g++ 12 is the safer compiler.
    APT_PACKAGES.append("g++-12")


def missing_packages(packages):
    out = subprocess.run(["dpkg-query", "-W", "-f", "${Package} ${Status}\n", *packages],
                         capture_output=True, text=True).stdout
    have = {line.split()[0] for line in out.splitlines() if line.endswith("install ok installed")}
    return [p for p in packages if p not in have]


t0 = time.time()
if not shutil.which("dpkg-query"):
    raise SystemExit("This notebook installs its packages with apt: it needs Debian or Ubuntu.")
todo = missing_packages(APT_PACKAGES)
if not todo:
    print("all packages present")
elif SUDO is None:
    print("cannot install packages here; missing:", " ".join(todo))
    print("the build or the run may fail; ask for them to be installed")
else:
    print("installing:", " ".join(todo))
    apt_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
    sh(SUDO + ["apt-get", "update", "-qq"], env=apt_env, show=lambda l: "Err" in l or "error" in l.lower())
    sh(SUDO + ["apt-get", "install", "-y", "-qq", "--no-install-recommends", *todo], env=apt_env,
       show=lambda l: l.startswith(("E:", "Setting up g++", "Setting up cmake", "Setting up mesa-vulkan")))
    left = missing_packages(APT_PACKAGES)
    print("done" if not left else f"still missing: {left}")
try:
    import PIL  # noqa: F401
except ImportError:
    sh([sys.executable, "-m", "pip", "install", "-q", "pillow"])
print(f"[{elapsed(t0)}]")

## 3. Rust
The pinned toolchain through rustup (`RUST_TOOLCHAIN`): the build cache is only valid for the
compiler that made it.

In [ ]:
t0 = time.time()
cargo_bin = CARGO_HOME / "bin"
os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"
if not shutil.which("rustup"):
    print("installing rustup")
    sh(f"curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal "
       f"--default-toolchain {shlex.quote(RUST_TOOLCHAIN)} --no-modify-path", quiet=True)
sh(["rustup", "toolchain", "install", RUST_TOOLCHAIN, "--profile", "minimal"], quiet=True)

# One environment for the build and for the run: `omnidroid play` runs `cargo test`, and a
# different compiler variable there would rebuild dynarmic.
BUILD_ENV = dict(os.environ, RUSTUP_TOOLCHAIN=RUST_TOOLCHAIN, CARGO_TERM_COLOR="never")
BUILD_ENV.pop("OMNIDROID_DYNARMIC_BUILD_DIR", None)
if BUILD_JOBS > 0:  # fewer parallel compilers: less memory
    BUILD_ENV["CARGO_BUILD_JOBS"] = str(BUILD_JOBS)
if shutil.which("g++-12") and DISTRO == "ubuntu22.04":
    BUILD_ENV.update(CC="gcc-12", CXX="g++-12")
print(subprocess.run(["cargo", "--version"], env=BUILD_ENV, capture_output=True, text=True).stdout.strip(),
      f"[{elapsed(t0)}]")

## 4. Source (and the build cache)
Clones `BRANCH` of `REPO_URL` (a `GITHUB_TOKEN` secret is used for a private repository), or
unpacks `SOURCE_TARBALL`. With `USE_BUILD_CACHE`, a cache of the source *and* its build made
earlier for the same commit and toolchain is restored instead, and the build cell has nothing
left to do.

In [ ]:
t0 = time.time()


def cache_dirs():
    """(where a cache is looked for, where one is written)."""
    if PLATFORM == "colab":
        if MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive")
        d = "/content/drive/MyDrive/omnidroid/cache"
        return ([d], d) if os.path.isdir("/content/drive/MyDrive") else ([], None)
    if PLATFORM == "kaggle":
        return (glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*"), "/kaggle/working")
    if PLATFORM == "modal":
        mounts = [m for m in glob.glob("/mnt/*") if os.path.isdir(m) and os.access(m, os.W_OK)]
        return ([f"{m}/cache" for m in mounts] + mounts, f"{mounts[0]}/cache" if mounts else None)
    d = str(WORK.parent / "cache")
    return ([d], d)


token = get_secret(GITHUB_TOKEN_SECRET) if GITHUB_TOKEN_SECRET else None
auth_url = REPO_URL
if token and REPO_URL.startswith("https://"):
    auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)


def remote_commit():
    if SOURCE_TARBALL:
        return None
    out = subprocess.run(["git", "ls-remote", auth_url, f"refs/heads/{BRANCH}"], capture_output=True,
                         text=True, timeout=120)
    if out.returncode != 0 or not out.stdout.strip():
        raise SystemExit(f"branch {BRANCH!r} not found at {REPO_URL} (private? set a {GITHUB_TOKEN_SECRET} "
                         f"secret) -- git said: {out.stderr.strip()[:300].replace(token or '@@', '***')}")
    return out.stdout.split()[0]


def local_commit():
    out = subprocess.run(["git", "-C", str(SRC), "rev-parse", "HEAD"], capture_output=True, text=True)
    return out.stdout.strip() if out.returncode == 0 else None


COMMIT = remote_commit()
CACHE_NAME = f"omnidroid-build-{(COMMIT or 'tarball')[:12]}-rust{RUST_TOOLCHAIN}-{DISTRO}.tar.zst"
CACHE_LOOK, CACHE_SAVE = cache_dirs() if USE_BUILD_CACHE else ([], None)
RESTORED = False
if COMMIT and local_commit() == COMMIT:
    print(f"{SRC} is already at {COMMIT[:12]}")
else:
    cached = next((p for d in CACHE_LOOK for p in glob.glob(f"{d}/{CACHE_NAME}")), None)
    SRC.parent.mkdir(parents=True, exist_ok=True)
    if SRC.exists():
        shutil.rmtree(SRC)
    if cached:
        print(f"restoring the build cache {cached} ({os.path.getsize(cached) / 2**30:.2f} GiB)")
        sh(["tar", "-I", "zstd -d -T0", "-xf", cached, "-C", "/"])
        RESTORED = True
    elif SOURCE_TARBALL:
        tmp = WORK / "src-unpack"
        shutil.rmtree(tmp, ignore_errors=True)
        tmp.mkdir(parents=True)
        sh(["tar", "-xf", SOURCE_TARBALL, "-C", str(tmp)])
        inner = list(tmp.iterdir())
        shutil.move(str(inner[0] if len(inner) == 1 and inner[0].is_dir() else tmp), str(SRC))
    else:
        print(f"cloning {REPO_URL} {BRANCH} ({COMMIT[:12]})")
        sh(["git", "clone", "--depth", "1", "--branch", BRANCH, auth_url, str(SRC)], quiet=True)
        sh(["git", "-C", str(SRC), "remote", "set-url", "origin", REPO_URL])  # no token left on disk
print(f"source at {SRC} ({local_commit() or 'not a git checkout'}) [{elapsed(t0)}]")

## 5. Build
`omnidroid` (the launcher) and the runtime it starts. The first build takes ~5-15 minutes
(4 cores: ~6); after a restored cache it takes seconds. A fresh build is then saved as the cache.

In [ ]:
t0 = time.time()
FRESH_BUILD = not (SRC / "target" / "release" / "omnidroid").exists()
show_cargo = lambda l: l.lstrip().startswith(("Compiling", "Finished", "Executable", "error"))
sh(["cargo", "build", "--release", "-p", "omnidroid"], env=BUILD_ENV, cwd=SRC, show=show_cargo)
sh(["cargo", "test", "--release", "-p", "omni-android", "--test", "gameactivity", "--no-run"],
   env=BUILD_ENV, cwd=SRC, show=show_cargo)
OMNIDROID = SRC / "target" / "release" / "omnidroid"
USAGE = subprocess.run([str(OMNIDROID), "--help"], capture_output=True, text=True).stdout
if "--no-window" not in USAGE or "--control" not in USAGE:
    raise SystemExit(f"this build ({BRANCH} {COMMIT and COMMIT[:12]}) has no --no-window/--control: headless mode "
                     "is in `unified` from dce9ac0 on. Set BRANCH to a branch that has it.")
print(f"built [{elapsed(t0)}]; the launcher has --no-window and --control")

if USE_BUILD_CACHE and CACHE_SAVE and not RESTORED and COMMIT and FRESH_BUILD:
    t1 = time.time()
    os.makedirs(CACHE_SAVE, exist_ok=True)
    target = f"{CACHE_SAVE}/{CACHE_NAME}"
    rel = str(SRC).lstrip("/")
    sh(["tar", "-I", "zstd -T0 -3", "-cf", target + ".part", "-C", "/",
        f"--exclude={rel}/target/omni-elf-fixtures", f"--exclude={rel}/target/debug", rel])
    os.replace(target + ".part", target)
    print(f"build cache saved: {target} ({os.path.getsize(target) / 2**30:.2f} GiB) [{elapsed(t1)}]")
    if PLATFORM == "kaggle":
        print("Kaggle: 'Save Version', then make a dataset from this notebook's output and add it as an "
              "input -- the next run restores it from /kaggle/input.")

## 6. The APK and the account
The APK is copied next to the run (a Drive or Volume mount is too slow to run from) and must be
the stock build: its sha256 is checked against `APK_SHA256`, and any other file is refused. The cookie
goes into a file only this user can read; it is never printed. The machine's exit country is
checked against `ACCOUNT_COUNTRY`.

In [ ]:
t0 = time.time()


def find_apk():
    if APK_PATH:
        return APK_PATH
    if PLATFORM == "colab" and MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    patterns = {
        "colab": ["/content/drive/MyDrive/omnidroid/*.apk", "/content/drive/MyDrive/*.apk", "/content/*.apk"],
        "kaggle": ["/kaggle/input/*/*.apk", "/kaggle/input/*/*/*.apk", "/kaggle/working/*.apk"],
        "modal": ["/mnt/*/*.apk", "/mnt/*/*/*.apk", "/root/*.apk"],
    }.get(PLATFORM, []) + [f"{WORK}/*.apk", f"{WORK.parent}/*.apk", f"{HOME}/*.apk"]
    found = [p for pat in patterns for p in sorted(glob.glob(pat)) if not p.startswith(str(WORK / "apk"))]
    if not found:
        raise SystemExit("No APK found. Put the Roblox APK where the table at the top says, or set APK_PATH. "
                         "Searched: " + ", ".join(patterns))
    stock = [p for p in found if sha256_of(p) in ALLOWED_APKS]
    return stock[0] if stock else found[0]


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


ALLOWED_APKS = {h.strip().lower() for h in APK_SHA256.split(",") if h.strip()}
apk_source = find_apk()
(WORK / "apk").mkdir(exist_ok=True)
APK = WORK / "apk" / os.path.basename(apk_source)
if not APK.exists() or APK.stat().st_size != os.path.getsize(apk_source):
    shutil.copyfile(apk_source, APK)
APK_DIGEST = sha256_of(APK)  # the file that runs, after the copy
if APK_DIGEST not in ALLOWED_APKS:
    APK.unlink()
    raise SystemExit(f"REFUSED: {apk_source} has sha256 {APK_DIGEST}, not the stock build's "
                     f"{' or '.join(sorted(ALLOWED_APKS)) or '(APK_SHA256 is empty)'}. A modified APK run with "
                     "a cookie can take the account. Use the stock APK from the Play Store.")
print(f"APK: {apk_source} ({APK.stat().st_size / 2**20:.0f} MiB), sha256 {APK_DIGEST} -- the stock build")
print(subprocess.run([str(OMNIDROID), "which", "--apk", str(APK)], capture_output=True, text=True).stdout.strip())

cookie = open(COOKIE_FILE).read() if COOKIE_FILE else get_secret(COOKIE_SECRET)
if not cookie or len(cookie.strip()) < 100:
    raise SystemExit(f"No cookie: set the {COOKIE_SECRET!r} secret (see the table at the top) or COOKIE_FILE.")
secrets_dir = WORK / "secrets"
secrets_dir.mkdir(mode=0o700, exist_ok=True)
COOKIE_PATH = secrets_dir / f"{ACCOUNT_NAME}.txt"
fd = os.open(COOKIE_PATH, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
with os.fdopen(fd, "w") as f:
    f.write(cookie.strip() + "\n")
print(f"cookie: {len(cookie.strip())} characters, in {COOKIE_PATH} (mode 600; not shown)")
del cookie

try:
    import urllib.request
    trace = urllib.request.urlopen("https://www.cloudflare.com/cdn-cgi/trace", timeout=15).read().decode()
    EXIT_COUNTRY = dict(l.split("=", 1) for l in trace.splitlines() if "=" in l).get("loc", "?")
except Exception as error:
    EXIT_COUNTRY = "?"
    print(f"(could not look up the exit country: {error})")
print(f"this machine reaches the internet from: {EXIT_COUNTRY}")
if ACCOUNT_COUNTRY and EXIT_COUNTRY != "?" and EXIT_COUNTRY.upper() != ACCOUNT_COUNTRY.upper():
    raise SystemExit(f"ACCOUNT_COUNTRY is {ACCOUNT_COUNTRY} and this machine is in {EXIT_COUNTRY}: Roblox may "
                     "end this cookie's session for good. Stopped before using it (clear ACCOUNT_COUNTRY to go on).")
if not ACCOUNT_COUNTRY:
    print("⚠️  If this account was not made/used from here, Roblox may end its session (see the top).")
print(f"[{elapsed(t0)}]")

## 7. Graphics: GPU or CPU
With an NVIDIA GPU whose EGL library the platform exposes, and a build with `--headless`, the
game renders on the GPU (an EGL vendor file points glvnd at `libEGL_nvidia.so.0`, as on
Colab). Otherwise Mesa llvmpipe renders on the CPU. Vulkan is limited to lavapipe, which the
engine refuses by its own rule, so it always takes its GLES path. A small EGL probe says which
renderer a headless context gets.

In [ ]:
t0 = time.time()
gpu_names = []
if shutil.which("nvidia-smi"):
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True)
    gpu_names = [l.strip() for l in out.stdout.splitlines() if l.strip()] if out.returncode == 0 else []
print("GPU:", "; ".join(gpu_names) if gpu_names else "none")


def find_lib(name):
    out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
    for line in out.splitlines():
        if line.strip().startswith(name + " ") and "x86-64" in line:
            return line.split("=>")[-1].strip()
    for d in ("/usr/lib64-nvidia", "/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
        if os.path.exists(f"{d}/{name}"):
            return f"{d}/{name}"
    return None


nvidia_egl = find_lib("libEGL_nvidia.so.0") if gpu_names else None
if gpu_names and not nvidia_egl:
    print("the GPU's EGL library (libEGL_nvidia.so.0) is not exposed here: rendering on the CPU")
USE_GPU = RENDERER != "cpu" and bool(nvidia_egl)
if RENDERER == "gpu" and not USE_GPU:
    print("RENDERER='gpu' asked, but there is no usable NVIDIA EGL here: using the CPU")

GFX_ENV = {}
mesa_json = next(iter(glob.glob("/usr/share/glvnd/egl_vendor.d/*mesa*.json")), None)
lvp = sorted(glob.glob("/usr/share/vulkan/icd.d/lvp_icd*.json"))
if lvp:  # lavapipe only: the engine skips an emulated Vulkan device and uses GLES
    GFX_ENV.update(VK_DRIVER_FILES=":".join(lvp), VK_ICD_FILENAMES=":".join(lvp))
if USE_GPU:
    vendor_json = WORK / "egl" / "10_nvidia.json"
    vendor_json.parent.mkdir(exist_ok=True)
    vendor_json.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": nvidia_egl}}))
    GFX_ENV["__EGL_VENDOR_LIBRARY_FILENAMES"] = ":".join(filter(None, [str(vendor_json), mesa_json]))
    GFX_ENV["LD_LIBRARY_PATH"] = ":".join(filter(None, [os.path.dirname(nvidia_egl), os.environ.get("LD_LIBRARY_PATH")]))
else:
    GFX_ENV.update(LIBGL_ALWAYS_SOFTWARE="1", GALLIUM_DRIVER="llvmpipe")
    if mesa_json:
        GFX_ENV["__EGL_VENDOR_LIBRARY_FILENAMES"] = mesa_json

EGL_PROBE = r'''
import ctypes as C, sys
egl = C.CDLL("libEGL.so.1"); gles = C.CDLL("libGLESv2.so.2")
egl.eglGetProcAddress.restype = C.c_void_p; egl.eglGetProcAddress.argtypes = [C.c_char_p]
def proc(name, res, *args):
    p = egl.eglGetProcAddress(name.encode())
    return C.CFUNCTYPE(res, *args)(p) if p else None
for fn, res, args in [("eglInitialize", C.c_uint, [C.c_void_p, C.c_void_p, C.c_void_p]),
                      ("eglBindAPI", C.c_uint, [C.c_uint]),
                      ("eglChooseConfig", C.c_uint, [C.c_void_p, C.POINTER(C.c_int), C.POINTER(C.c_void_p), C.c_int, C.POINTER(C.c_int)]),
                      ("eglCreatePbufferSurface", C.c_void_p, [C.c_void_p, C.c_void_p, C.POINTER(C.c_int)]),
                      ("eglCreateContext", C.c_void_p, [C.c_void_p, C.c_void_p, C.c_void_p, C.POINTER(C.c_int)]),
                      ("eglMakeCurrent", C.c_uint, [C.c_void_p, C.c_void_p, C.c_void_p, C.c_void_p]),
                      ("eglTerminate", C.c_uint, [C.c_void_p])]:
    getattr(egl, fn).restype = res; getattr(egl, fn).argtypes = args
gles.glGetString.restype = C.c_char_p; gles.glGetString.argtypes = [C.c_uint]
get_display = proc("eglGetPlatformDisplayEXT", C.c_void_p, C.c_uint, C.c_void_p, C.c_void_p)
def renderer(dpy):
    if not dpy or not egl.eglInitialize(dpy, None, None): return None
    egl.eglBindAPI(0x30A0)
    ints = lambda *v: (C.c_int * len(v))(*v)
    cfg = C.c_void_p(); n = C.c_int()
    if not egl.eglChooseConfig(dpy, ints(0x3033, 1, 0x3040, 0x40, 0x3038), C.byref(cfg), 1, C.byref(n)) or not n.value: return None
    surf = egl.eglCreatePbufferSurface(dpy, cfg, ints(0x3057, 16, 0x3056, 16, 0x3038))
    ctx = egl.eglCreateContext(dpy, cfg, None, ints(0x3098, 3, 0x3038))
    if not surf or not ctx or not egl.eglMakeCurrent(dpy, surf, surf, ctx): return None
    return (gles.glGetString(0x1F01) or b"?").decode()
found = []
if sys.argv[1] == "gpu":
    query = proc("eglQueryDevicesEXT", C.c_uint, C.c_int, C.POINTER(C.c_void_p), C.POINTER(C.c_int))
    devices = (C.c_void_p * 16)(); n = C.c_int()
    if query and query(16, devices, C.byref(n)):
        for i in range(n.value):
            r = renderer(get_display(0x313F, devices[i], None))
            if r: found.append(r)
else:
    r = renderer(get_display(0x31DD, None, None))  # EGL_PLATFORM_SURFACELESS_MESA
    if r: found.append(r)
print("; ".join(found) if found else "no EGL context")
'''
probe = subprocess.run([sys.executable, "-c", EGL_PROBE, "gpu" if USE_GPU else "cpu"],
                       env=dict(os.environ, **GFX_ENV), capture_output=True, text=True, timeout=120)
EGL_RENDERER = (probe.stdout.strip() or probe.stderr.strip()[-300:]) if probe.returncode == 0 else \
    f"probe failed: {probe.stderr.strip()[-300:]}"
print(f"no display; rendering on the {'GPU (NVIDIA EGL)' if USE_GPU else 'CPU (Mesa llvmpipe)'}")
print(f"a headless EGL context here gets: {EGL_RENDERER}   [{elapsed(t0)}]")

## 8. Start the game
`omnidroid play --no-window --control <file>` in the background: no display, headless (drawing
off) unless `START_HEADLESS` is off. Everything it prints goes to `game.log`.

In [ ]:
STATE = globals().get("STATE", {})


def running():
    p = STATE.get("proc")
    return p is not None and p.poll() is None


if APK_DIGEST not in ALLOWED_APKS:
    raise SystemExit("REFUSED: the APK is not the stock build (see the APK cell)")
if running():
    print("already running (stop it with the last cell first)")
else:
    env = dict(BUILD_ENV, **GFX_ENV)
    env.pop("WAYLAND_DISPLAY", None)
    env["OMNI_WINDOW_SIZE"] = WINDOW_SIZE
    if GRAPHICS_QUALITY:
        env["OMNI_GRAPHICS_QUALITY"] = str(GRAPHICS_QUALITY)
    join_delay = JOIN_DELAY or (20 if USE_GPU else 45)
    data_dir = WORK / "data" / ACCOUNT_NAME
    data_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(OMNIDROID), "play", "--apk", str(APK), "--cookie", str(COOKIE_PATH), "--place", str(PLACE_ID),
           "--join-delay", str(join_delay), "--minutes", str(SESSION_MINUTES), "--data-dir", str(data_dir)]
    control = WORK / "control.txt"
    control.write_text("")
    # No display: an off-screen EGL surface (NVIDIA's EGL device, or Mesa's surfaceless platform).
    cmd += ["--no-window", "--control", str(control)]
    if not START_HEADLESS:
        control.write_text("headless off\n")
    for name in ("DISPLAY", "XAUTHORITY", "WAYLAND_DISPLAY"):
        env.pop(name, None)
    # LAUNCH_PREFIX, e.g. "nice -n 5" or "flock /tmp/game.lock": a command the launch runs under.
    cmd = shlex.split(LAUNCH_PREFIX) + cmd
    LOG = WORK / "game.log"
    STATE.update(log=LOG, control=control, headless=START_HEADLESS, started=time.time(), shots=0,
                 proc=subprocess.Popen(cmd, env=env, cwd=WORK, stdin=subprocess.DEVNULL,
                                       stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True))
    print("started:", shlex.join(cmd).replace(str(COOKIE_PATH), "<cookie file>"))
    print(f"log: {LOG}   control: {control}")

## 9. Wait until the game is on screen
Follows the log: the runtime, the GLES context, the sign-in (`DID_LOG_IN`), the join, and the
engine's own `onGameLoaded: placeId:<PLACE_ID>`. After that the game still shows its own
loading screen while the world streams in (a minute or two on a CPU), so from `SETTLE_SECONDS`
after the load it looks at the screen every 20 s and waits until it is no longer one flat
colour -- a loading screen is ~95% one colour, the world ~20%.

In [ ]:
from IPython.display import HTML, Image as IPyImage, display

FRAMES_RE = re.compile(r"^FRAMES: \+(\d+)s into the session, (\d+) presents \(\+(\d+) in the last (\d+)s\)")


def log_lines(start=0):
    with open(STATE["log"], errors="replace") as f:
        f.seek(start)
        text = f.read()
        return text.splitlines(), f.tell()


def control(command, expect=None, timeout=60):
    """Appends `command` to the control file; with `expect`, waits for a log line containing it."""
    start = os.path.getsize(STATE["log"])
    with open(STATE["control"], "a") as f:
        f.write(command + "\n")
    if not expect:
        return None
    deadline = time.time() + timeout
    while time.time() < deadline:
        lines, _ = log_lines(start)
        for line in lines:
            if any(e in line for e in ([expect] if isinstance(expect, str) else expect)):
                return line.strip()
        if not running():
            raise RuntimeError("the game has exited")
        time.sleep(0.5)
    raise TimeoutError(f"no {expect!r} within {timeout}s after {command!r}")


def take_screenshot(path, say=print):
    """The game's screen as a PNG at `path`: `screenshot <path>` through the control file (the
    frame is rendered for real even while headless)."""
    answer = control(f"screenshot {path}", ("SCREENSHOT: saved", "SCREENSHOT: failed"), timeout=120)
    if "failed" in answer and STATE["headless"]:  # once more with drawing switched on
        say(answer)
        say(control("headless off", "CONTROL: headless"))
        time.sleep(5)
        answer = control(f"screenshot {path}", ("SCREENSHOT: saved", "SCREENSHOT: failed"), timeout=120)
        say(control("headless on", "CONTROL: headless"))
    say(answer)
    if "failed" in answer:
        raise RuntimeError(answer)
    return Path(path)


def flatness(path):
    """The share of the screen in its most common colour (16 levels a channel)."""
    from PIL import Image
    img = Image.open(path).convert("RGB").resize((320, 180))
    colours = img.point(lambda v: v & 0xF0).getcolors(320 * 180)
    return max(count for count, _ in colours) / (320 * 180)


def wait_for_game(timeout_minutes=LOAD_TIMEOUT_MINUTES, settle=SETTLE_SECONDS):
    t0 = STATE["started"]
    # The clock starts when the game first prints: a LAUNCH_PREFIX (a lock, say) may hold it back.
    while os.path.getsize(STATE["log"]) == 0 and running():
        time.sleep(2)
    t0 = time.time() if time.time() - t0 > 10 else t0
    milestones = [
        ("DISPLAY:", "the runtime is up"),
        ("Initialized EGL context", "the engine has a GLES context"),
        ("GL Renderer:", None),
        ("DID_LOG_IN", "signed in"),
        ("JOIN: nativeAppBridgeV2StartGameWithParam", "joining"),
        (f"onGameLoaded: placeId:{PLACE_ID}", "the place has loaded; its loading screen is up"),
    ]
    seen, pos, loaded_at, frames, last_status, warned, next_look = set(), 0, None, None, 0, False, 0
    probe = SHOTS / "omnidroid-probe.png"
    while True:
        lines, pos = log_lines(pos)
        for line in lines:
            for marker, what in milestones:
                if marker in line and marker not in seen:
                    seen.add(marker)
                    shown = what or line.split("] ", 1)[-1].strip()
                    print(f"[{elapsed(t0)}] {shown}" + (f"  ({line.strip()[:120]})" if marker.startswith("JOIN") else ""))
                    if marker.startswith("onGameLoaded"):
                        loaded_at = time.time()
            m = FRAMES_RE.match(line)
            if m:
                frames = (int(m.group(2)), int(m.group(3)) / int(m.group(4)))
            if "GUEST THREAD DIED" in line or "panicked at" in line or "JOIN: start failed" in line:
                print(f"[{elapsed(t0)}] !! {line.strip()[:300]}")
        if not running():
            tail, _ = log_lines(max(0, os.path.getsize(STATE["log"]) - 6000))
            print("\n".join(tail[-40:]))
            raise RuntimeError(f"the game exited (code {STATE['proc'].returncode}) before it was on screen")
        now = time.time()
        if not warned and "DID_LOG_IN" not in seen and now - t0 > 180:
            warned = True
            print(f"[{elapsed(t0)}] ⚠️  not signed in after 3 minutes: the cookie may be expired or rejected "
                  "(another country?). Still waiting.")
        if now - last_status > 60 and frames:
            last_status = now
            print(f"[{elapsed(t0)}] ... {frames[0]} frames so far, {frames[1]:.1f}/s")
        if loaded_at and now - loaded_at >= settle and now >= next_look:
            next_look = now + 20
            share = flatness(take_screenshot(probe, say=lambda _: None))
            if share < 0.6:
                STATE["shots"] += 1
                ready = SHOTS / f"omnidroid-{STATE['shots']}.png"
                os.replace(probe, ready)
                STATE["ready_shot"] = ready
                print(f"[{elapsed(t0)}] in the world ({share:.0%} of the screen in one colour"
                      + (f", {frames[1]:.1f} frames/s)" if frames else ")"))
                return ready
            print(f"[{elapsed(t0)}] ... still a loading screen ({share:.0%} one colour)")
        if now - t0 > timeout_minutes * 60:
            if loaded_at:
                print(f"⚠️  the place loaded but the screen still looks like a loading screen after "
                      f"{timeout_minutes} minutes; the next cell shows it anyway")
                return None
            raise TimeoutError(f"the place did not load in {timeout_minutes} minutes; the log is {STATE['log']}")
        time.sleep(2)


STATE.pop("ready_shot", None)
wait_for_game()

## 10. Screenshot
The screen the previous cell found in the world, shown here and printed as a `data:` URL: copy
the whole line starting `data:image/jpeg;base64,` into a browser's address bar. The PNG and a
JPEG are also saved (Colab: the Files panel, `/content`; Kaggle: the notebook's output).
(It is taken with `screenshot <path>` in the control file: that one frame is rendered for real
even while headless.)

In [ ]:
def screenshot():
    STATE["shots"] += 1
    return take_screenshot(SHOTS / f"omnidroid-{STATE['shots']}.png")


def show(path, max_url_kb=400):
    from PIL import Image
    img = Image.open(path).convert("RGB")
    display(IPyImage(filename=str(path)))
    share = flatness(path)
    print(f"{path}  {img.width}x{img.height}" + (f"   ⚠️ {share:.0%} one colour: a loading screen?" if share >= 0.6 else ""))
    img.thumbnail((1280, 720))
    for quality in (80, 70, 60, 50, 40):
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=quality, optimize=True)
        url = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()
        if len(url) <= max_url_kb * 1024:
            break
    jpg = Path(path).with_suffix(".jpg")
    jpg.write_bytes(buf.getvalue())
    display(HTML(f'<a download="{jpg.name}" href="{url}">download {jpg.name}</a> ({len(url) // 1024} KB as a data: URL)'))
    print("Paste this whole line into a browser's address bar:")
    print(url)
    return url


SHOT = STATE.get("ready_shot") or screenshot()
DATA_URL = show(SHOT)

## 11. Drawing off / on
`headless on`: the game keeps running and presenting frames, but none is drawn (far less
CPU/GPU); `headless off` draws again. A screenshot renders its one frame either way.

In [ ]:
print(control("headless on", "CONTROL: headless"))
STATE["headless"] = True

In [ ]:
print(control("headless off", "CONTROL: headless"))
STATE["headless"] = False

## 12. Another screenshot

In [ ]:
show(screenshot())

## 13. How it ran, and stop
The frame rate and the engine's messages, then the game is stopped (interrupt, then kill). A
session also ends by itself after `SESSION_MINUTES`. Run the start cell again for a new one.

In [ ]:
frames = [FRAMES_RE.match(l) for l in log_lines()[0]]
rates = [int(m.group(3)) / int(m.group(4)) for m in frames if m]
if rates:
    print(f"frames/s, the last 15 s: {', '.join(f'{r:.1f}' for r in rates[-3:])}; the best 5 s: {max(rates):.1f}")
if running():
    os.killpg(STATE["proc"].pid, signal.SIGINT)
    try:
        STATE["proc"].wait(30)
    except subprocess.TimeoutExpired:
        os.killpg(STATE["proc"].pid, signal.SIGKILL)
        STATE["proc"].wait()
    print("stopped")
print(f"log: {STATE['log']}   screenshots: {SHOTS}")